## Production Deployment Checklist

### Database Optimization
- [ ] Add index on `users.api_key_hash` (critical for lookup performance)
- [ ] Add index on `permission_logs.timestamp` (for audit queries)
- [ ] Configure connection pool: 50-100 connections for production
- [ ] Enable connection pool pre-ping to avoid stale connections

### Caching Strategy
- [ ] Implement Redis for permission caching (TTL: 5 minutes)
- [ ] Cache user-role mappings to reduce database queries
- [ ] Invalidate cache on role assignment/revocation
- [ ] Monitor cache hit rate (target >80%)

### Monitoring & Alerts
- [ ] Track permission check latency P95 <100ms
- [ ] Monitor database connection pool utilization (<80%)
- [ ] Alert on 403 error rate >10%
- [ ] Log unusual role assignments for security review

### Security Hardening
- [ ] Enable rate limiting on role management endpoints
- [ ] Implement IP-based access logs for audit
- [ ] Use prepared statements to prevent SQL injection
- [ ] Regular security audits of role assignments

### Compliance Integration
- [ ] Forward permission logs to Elasticsearch (Module 6.4)
- [ ] Generate monthly access reports for compliance
- [ ] Implement log retention policy (7 years for SOC2)
- [ ] Document role definitions and approval process

## Next Steps

This module connects to:
- **Module 6.1**: Documents with PII auto-marked as \"confidential\"
- **Module 6.2**: Database credentials managed via secrets manager
- **Module 6.4**: Permission logs fed to compliance auditing system

**Practathon Challenges:**
- **Easy (60 min)**: Implement basic 3-role RBAC with permission testing
- **Medium (90-120 min)**: Add Redis caching and cycle detection
- **Hard (4-5 hours)**: Production deployment with monitoring and ELK integration

**Resources:**
- Casbin documentation: https://casbin.org/
- PostgreSQL RBAC patterns: https://www.postgresql.org/docs/current/ddl-rowsecurity.html
- Module 6.4 (Compliance & Auditing): Coming next!

## Decision Card: RBAC for RAG Systems

### ✅ Benefits
- **Enterprise security** for 10-5K users with document-level filtering
- **Compliance-ready** audit trails (SOC2/ISO27001 passing)
- **Flexible** role hierarchy (admin > editor > viewer)
- **Performance** overhead: 20-50ms per request

### ⚠️ Limitations
- Adds 20-50ms latency per permission check
- Cannot express complex conditional policies (use ABAC instead)
- Role changes affect next request, not cached data
- Scales to ~10K users (requires Redis optimization beyond that)

### 💰 Cost
- **Infrastructure**: $100-400/month (PostgreSQL $50-100, Redis $30-50, monitoring $20)
- **Implementation**: 8-12 hours initial setup
- **Maintenance**: 2-4 hours/month (role audits, policy updates)

### 🎯 Use When:
- 10-5,000 users
- Handle confidential documents with varying access levels
- Compliance requirements (audit logging needed)
- Budget allows $100+/month infrastructure
- Accept <100ms permission check overhead

### 🚫 Avoid When:
- <10 users (use simple admin flag)
- Need instant revocation across all sessions
- Complex attribute-based policies required
- >10K concurrent users (use managed identity service)
- Already using JWT-based microservices (embed roles in claims instead)

## When NOT to Use This Approach

RBAC adds complexity and overhead. **Avoid** in these scenarios:

### ❌ 1. Small Teams (<10 users, two-tier access)
**Why**: Boolean `is_admin` flag is simpler\n**Alternative**: Add `is_admin` column to users table\n**Overhead saved**: 20-50ms per request, database complexity

### ❌ 2. Complex Conditional Policies
**Why**: RBAC can't express \"allow if time < 6pm AND user.dept == doc.dept\"\n**Alternative**: Use ABAC (Attribute-Based Access Control) or OPA\n**Limitation**: RBAC only handles role-based rules, not contextual attributes

### ❌ 3. Need Instant Session-Wide Revocation
**Why**: Role changes affect next request, not active sessions\n**Alternative**: Session invalidation + JWT blacklisting\n**Limitation**: Cached permissions remain valid until refresh

### ❌ 4. >10,000 Concurrent Users
**Why**: Centralized PostgreSQL becomes bottleneck\n**Alternative**: Managed identity service (Auth0, Okta) with distributed caching\n**Scale limit**: ~5,000 users without Redis; ~10,000 with Redis

### ❌ 5. Multi-Tenant SaaS Requiring Tenant Isolation
**Why**: RBAC doesn't enforce tenant boundaries natively\n**Alternative**: Add tenant_id filtering layer before RBAC\n**Limitation**: Must combine with row-level security or tenant-scoped queries

## Alternative Solutions

RBAC is not always the best choice. Consider these alternatives:

### 1. Simple Admin/Non-Admin Flag
**Best for**: <10 users with two-tier access\n**Trade-off**: Zero overhead, but inflexible\n**Cost**: $0 additional\n**Example**: Add `is_admin` boolean to user table

### 2. JWT Claims-Based RBAC
**Best for**: Stateless microservices, mobile apps\n**Trade-off**: Harder to revoke tokens immediately\n**Cost**: $0 (built into JWT)\n**Example**: Embed `roles: ['admin', 'editor']` in JWT payload

### 3. Managed Identity Services (Auth0, Okta, Cognito)
**Best for**: Enterprise scale (>5K users), compliance requirements\n**Trade-off**: Vendor lock-in, higher cost\n**Cost**: $100-1000+/month\n**Example**: Outsource entire auth/RBAC to Auth0

### 4. Attribute-Based Access Control (ABAC/OPA)
**Best for**: Complex conditional policies (time-based, location-based)\n**Trade-off**: More complex to configure and debug\n**Cost**: $50-200/month (Open Policy Agent)\n**Example**: \"Allow if user.department == document.department AND time.hour < 18\"

In [ ]:
# INSECURE: Accepting client-provided filter (DO NOT DO THIS)
# def query_documents_insecure(query, client_provided_filter):
#     return pinecone_index.query(vector=embed(query), filter=client_provided_filter)

# SECURE: Generate filter server-side based on user's actual roles
def query_documents_secure(user, query_text):
    # Generate filter based on user's authenticated roles
    metadata_filter = rbac.get_pinecone_filter(user)
    print(f"Server-enforced filter for {user.username}: {metadata_filter}")
    # return pinecone_index.query(vector=embed(query_text), filter=metadata_filter)
    return metadata_filter

# Test with different users
print("Secure Pinecone Filtering:")
viewer_filter = query_documents_secure(viewer_user, "sensitive data")
admin_filter = query_documents_secure(admin_user, "sensitive data")

print(f"\nViewer can only access: {viewer_filter['access_level']['$in']}")
print(f"Admin can access: {admin_filter['access_level']['$in']}")

# Expected: Filters are server-generated and match user's actual permissions

### 2. Pinecone Filter Bypass (Client-Side Manipulation)

**Problem**: Trusting client-provided access level filters allows users to query confidential documents.

**Root Cause**: API accepts `access_level` parameter from client instead of enforcing server-side.

**Fix**: ALWAYS generate Pinecone filters server-side based on authenticated user's roles. Never trust client input for security decisions.

In [ ]:
# INSECURE: Role assignment without permission check (DO NOT DO THIS)
# def assign_role_insecure(username, role_name):
#     return rbac.assign_role(username, role_name)

# SECURE: Check that current user has admin permissions first
def assign_role_secure(current_user, username, role_name):
    # Verify current user has permission to manage roles
    if not rbac.check_permission(current_user, "role", "manage", log_check=True):
        raise PermissionError(f"User {current_user.username} cannot manage roles")
    return rbac.assign_role(username, role_name)

# Test: Viewer tries to assign admin role (should fail)
try:
    assign_role_secure(viewer_user, "charlie", "admin")
    print("✗ SECURITY ISSUE: Viewer was able to assign admin role!")
except PermissionError as e:
    print(f"✓ Security check passed: {e}")

# Expected: Permission denied for viewer attempting role management

## Common Production Failures

### 1. Permission Escalation via Unprotected Role Assignment

**Problem**: Missing authorization checks on admin endpoints allow any user to assign themselves admin role.

**Root Cause**: Role assignment endpoint lacks permission check requiring admin role.

**Fix**: Always wrap role management endpoints with `require_permission("role", "manage")`

In [ ]:
# Perform permission checks with logging enabled
rbac.check_permission(admin_user, "document", "read", log_check=True)
rbac.check_permission(editor_user, "document", "delete", log_check=True)  # Will be denied
rbac.check_permission(viewer_user, "document", "write", log_check=True)   # Will be denied

# Get permission statistics
stats = rbac.get_permission_stats(hours=24)
print(f"Permission Check Statistics (last 24h):")
print(f"  Total checks: {stats.get('total_checks', 0)}")
print(f"  Denied checks: {stats.get('denied_count', 0)}")
print(f"  Denial rate: {stats.get('denied_rate', 0):.1%}")

# Expected: Stats showing total checks and denial rate

## Audit Logging

Every permission check is logged for compliance (SOC2, ISO27001):
- User ID
- Resource and action
- Timestamp
- Allowed/denied result
- Optional IP address

Logs feed into Module 6.4 compliance reporting.

In [ ]:
# Promote viewer to editor
print("Promoting charlie from viewer to editor...")
success = rbac.assign_role("charlie", "editor")
print(f"Success: {success}")

# Retrieve updated user
updated_charlie = rbac.get_user_by_api_key("viewer-key-789")
print(f"Charlie's roles: {updated_charlie.get_role_names()}")
print(f"Charlie's accessible levels: {rbac.get_accessible_levels(updated_charlie)}")

# Revoke editor role
print("\nRevoking editor role from charlie...")
success = rbac.revoke_role("charlie", "editor")
print(f"Success: {success}")

updated_charlie = rbac.get_user_by_api_key("viewer-key-789")
print(f"Charlie's roles after revocation: {updated_charlie.get_role_names()}")

# Expected: Charlie gains editor permissions, then loses them after revocation

## Role Management

Users can be assigned multiple roles, and roles can be added or revoked dynamically.

**Important**: Role changes take effect immediately on the next request. Already-cached data in the client is not affected.

In [ ]:
# Get accessible document levels for each user
print("Document Access Levels:")

admin_levels = rbac.get_accessible_levels(admin_user)
print(f"\nAdmin (alice): {admin_levels}")

editor_levels = rbac.get_accessible_levels(editor_user)
print(f"Editor (bob): {editor_levels}")

viewer_levels = rbac.get_accessible_levels(viewer_user)
print(f"Viewer (charlie): {viewer_levels}")

# Generate Pinecone metadata filters
print("\nPinecone Filters:")
print(f"Admin filter: {rbac.get_pinecone_filter(admin_user)}")
print(f"Editor filter: {rbac.get_pinecone_filter(editor_user)}")
print(f"Viewer filter: {rbac.get_pinecone_filter(viewer_user)}")

# Expected: Admin sees all 3 levels, Editor sees public+internal, Viewer sees public only

## Document-Level Access Control

Documents are classified into three access levels stored as Pinecone metadata:

- **Public**: Accessible by all roles (viewer, editor, admin)
- **Internal**: Accessible by editor and admin only
- **Confidential**: Accessible by admin only

The RBAC manager generates Pinecone filters to enforce access control server-side.

In [ ]:
# Test permission checks for different users
print("Permission Checks:")
print(f"\nAdmin (alice):")
print(f"  Can read: {rbac.check_permission(admin_user, 'document', 'read', log_check=False)}")
print(f"  Can write: {rbac.check_permission(admin_user, 'document', 'write', log_check=False)}")
print(f"  Can delete: {rbac.check_permission(admin_user, 'document', 'delete', log_check=False)}")
print(f"  Can manage users: {rbac.check_permission(admin_user, 'user', 'manage', log_check=False)}")

print(f"\nEditor (bob):")
print(f"  Can read: {rbac.check_permission(editor_user, 'document', 'read', log_check=False)}")
print(f"  Can write: {rbac.check_permission(editor_user, 'document', 'write', log_check=False)}")
print(f"  Can delete: {rbac.check_permission(editor_user, 'document', 'delete', log_check=False)}")

print(f"\nViewer (charlie):")
print(f"  Can read: {rbac.check_permission(viewer_user, 'document', 'read', log_check=False)}")
print(f"  Can write: {rbac.check_permission(viewer_user, 'document', 'write', log_check=False)}")

# Expected: Admin has all permissions, Editor has read/write, Viewer has read only

## Permission Checking

Casbin enforces permissions based on role hierarchy:
- **Viewer**: Can read documents
- **Editor**: Can read + write documents (inherits from viewer)
- **Admin**: Can read + write + delete documents + manage users/roles (inherits from editor)

Permission checks add ~20-50ms latency per request.

In [ ]:
# Create users with different roles
admin_user = rbac.create_user("alice", "admin-key-123", "alice@example.com", ["admin"])
editor_user = rbac.create_user("bob", "editor-key-456", "bob@example.com", ["editor"])
viewer_user = rbac.create_user("charlie", "viewer-key-789", "charlie@example.com", ["viewer"])

print("Created users:")
print(f"  Admin: {admin_user.username} (roles: {admin_user.get_role_names()})")
print(f"  Editor: {editor_user.username} (roles: {editor_user.get_role_names()})")
print(f"  Viewer: {viewer_user.username} (roles: {viewer_user.get_role_names()})")

# Expected: 3 users created with appropriate role assignments

## Create Users with Roles

Users are created with:
- Unique username and email
- API key (stored as SHA-256 hash)
- One or more assigned roles

API keys are hashed for security (SHA-256). Never store plaintext keys in production.

In [ ]:
# Initialize RBAC Manager with SQLite (for demo)
database_url = "sqlite:///rbac_demo.db"

rbac = RBACManager(database_url)

print("✓ RBAC Manager initialized")
print(f"  Database: {database_url}")
print(f"  Casbin model: {rbac.model_path}")

# Expected: RBAC Manager initialized with database and Casbin policy model

## Initialize RBAC Manager

The RBAC Manager coordinates all access control operations:
- Database connection with pooling (20 connections, 30 overflow)
- Casbin policy enforcement
- Default role hierarchy setup
- Permission audit logging

We'll use SQLite for this demo (use PostgreSQL in production).

In [ ]:
# Install required packages (if not already installed)
# !pip install -r requirements.txt

# Import necessary modules
import sys
import os
import json
import importlib.util

# Import the RBAC module
spec = importlib.util.spec_from_file_location("l2_m6_rbac", "l2_m6_rbac_multi-level_access.py")
rbac_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(rbac_module)

RBACManager = rbac_module.RBACManager
User = rbac_module.User
Role = rbac_module.Role
AccessLevel = rbac_module.AccessLevel
RoleName = rbac_module.RoleName

print("✓ Imports successful")

## Setup & Installation

### Install Dependencies

First, ensure all required packages are installed:

# Module 6.3: RBAC & Multi-Level Access Control

## Overview

This module teaches role-based access control (RBAC) implementation for enterprise RAG systems.

**Key Learning Outcomes:**
- Implement three-tier RBAC (admin, editor, viewer) with measurable permission differences
- Persist user-role mappings in PostgreSQL with proper relationship models
- Enforce document-level access control using Pinecone metadata filtering
- Filter queries in real-time with <100ms permission overhead
- Understand when RBAC is appropriate vs. simpler or more complex alternatives

**Prerequisites:**
- Level 1 Module 3.3 (API authentication)
- M6.1 (PII detection)
- M6.2 (secrets management)

**Duration:** 55 minutes

## Core Concepts

### What is RBAC?

**Role-Based Access Control (RBAC)** assigns permissions to roles rather than individual users. Users are assigned one or more roles, inheriting their permissions.

**Five-Component Foundation:**

1. **Casbin Policy Model** - Defines role hierarchy and permission matching rules
2. **Database Schema** - User, Role, and Permission tables with many-to-many relationships
3. **RBAC Manager** - Core class handling policy enforcement and user-role assignments
4. **FastAPI Middleware** - Integrates permission checking into request processing
5. **Pinecone Filtering** - Restricts query results based on user's accessible document levels

### Permission Structure

Three access levels enforced in our implementation:

| Level | Viewer | Editor | Admin |
|-------|--------|--------|-------|
| **Public** | ✓ Read | ✓ Read/Write | ✓ Read/Write/Delete |
| **Internal** | ✗ | ✓ Read/Write | ✓ Read/Write/Delete |
| **Confidential** | ✗ | ✗ | ✓ Read/Write/Delete |

**Role Hierarchy:** admin > editor > viewer (inheritance model)